# 머신러닝 파일별 보강3 · 12 앙상블 ② Boosting · AdaBoost · Gradient Boosting

**한 줄 요약**

- Bagging이 분산을 줄였다면 Boosting은 편향을 줄인다 · 회귀 트리 · 잔차를 순차로 보정 · AdaBoost
- Gradient Boosting · 손실 함수 · XGBoost · LightGBM
- CatBoost 계열을 Wine으로

- **원본** · `6.머닝러신/06_앙상블/12_ensemble_part2 .ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/05_머신러닝` 종합 정리 범위 밖 (파일별 7 Bagging · RF 다음)

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. 회귀 트리가 구간별 평균을 예측하고, 부스팅이 잔차를 순차로 줄이는 것을 숫자로 보인다
2. AdaBoost의 n_estimators · learning_rate를 train 안의 CV로 고르고 test는 한 번만 본다
3. 경사하강의 학습률 크기에 따른 수렴 · 발산과 Gradient Boosting의 관계를 말한다
4. 이상치가 있을 때 squared_error · absolute_error · huber의 차이를 '진짜 함수와의 거리'로 판단한다
5. 히스토그램(bin) · 범주형 처리 · 얕은 트리 vs 깊은 트리를 실험 근거로 설명한다

## 어디에 쓰나

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">실제 상황</th>
<th style="text-align: left;">이 권에서 배우는 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">단순한 모델이 계속 덜 맞을 때 (과소적합)</td>
<td style="text-align: left;">Boosting으로 편향 줄이기 (2 · 3장)</td>
</tr>
<tr>
<td style="text-align: left;">이상치가 섞인 회귀 문제</td>
<td style="text-align: left;">손실 함수 선택 (4장)</td>
</tr>
<tr>
<td style="text-align: left;">데이터가 크고 빠른 학습이 필요할 때</td>
<td style="text-align: left;">히스토그램 기반 부스팅 (5-1)</td>
</tr>
<tr>
<td style="text-align: left;">범주형 열이 많은 표</td>
<td style="text-align: left;">원-핫 vs 네이티브 범주 처리 (5-2)</td>
</tr>
</tbody>
</table>

## 학습 지도

<img src="images/pfm10_map.png" alt="12 앙상블 ② 학습 지도" width="763">

**예상 학습 시간** · 개념 1시간 30분 + 실습 1시간 (전체 실행 2분 안팎 · GPU 필요 없음)

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">장</th>
<th style="text-align: left;">제목</th>
<th style="text-align: left;">이 장의 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">0</td>
<td style="text-align: left;">준비</td>
<td style="text-align: left;">Wine · 평가 계약 · 원본 목표</td>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">회귀 트리와 잔차</td>
<td style="text-align: left;">트리는 숫자를 어떻게 예측하고, 잔차를 순차로 맞추면 오차는 어떻게 줄어드는가?</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">AdaBoost</td>
<td style="text-align: left;">AdaBoost는 틀린 샘플에 어떻게 집중하고, 스텝 수 · 학습률은 어떻게 고르는가?</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">Gradient Boosting</td>
<td style="text-align: left;">Gradient Boosting은 AdaBoost와 무엇이 다르고, 학습률은 왜 중요한가?</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">손실 함수</td>
<td style="text-align: left;">Gradient Boosting의 손실 함수를 바꾸면 무엇이 달라지고, 이상치가 있을 때는 무엇을 고르는가?</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">XGBoost · LightGBM · CatBoost</td>
<td style="text-align: left;">히스토그램(bin) 수 · 범주형 처리 · 라이브러리 차이는 결과를 어떻게 바꾸는가?</td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;">종합 실습</td>
<td style="text-align: left;">Boosting은 왜 얕은 트리를, Random Forest는 왜 깊은 트리를 써도 괜찮은가? (원본 숙제 Q2 · Q4)</td>
</tr>
<tr>
<td style="text-align: left;">—</td>
<td style="text-align: left;">마무리 · 부록</td>
<td style="text-align: left;">오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td>
</tr>
</tbody>
</table>

---
## 0. 준비

- 원본은 CV
- GridSearch를 **전체 데이터**로 돌린 뒤 같은 데이터의 test 점수를 보고함
- test가 선택에 섞임.
- 이 노트북은 test를 먼저 봉인하고 CV를 train 안에서만 함.
- **셀은 위에서부터 차례로 실행**.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">항목</th>
<th style="text-align: left;">설정</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">라이브러리</td>
<td style="text-align: left;">scikit-learn · xgboost (lightgbm · catboost · graphviz는 이 환경에 없음 → 5장에서 sklearn 대체)</td>
</tr>
<tr>
<td style="text-align: left;">seed</td>
<td style="text-align: left;"><code>SEED = 42</code> · CV = StratifiedKFold(5, shuffle=True, 42)</td>
</tr>
<tr>
<td style="text-align: left;">데이터</td>
<td style="text-align: left;">Wine 178 → train 142 · test 36 · sin 회귀 80개 · 범주형 가짜 데이터 200개</td>
</tr>
<tr>
<td style="text-align: left;">원본 그림</td>
<td style="text-align: left;">슬라이드 그림 18장(png 파일이 원본 폴더에 없음) · plotly 그림 9개 → 장 그림 · 결과 줄로</td>
</tr>
</tbody>
</table>

In [1]:
# 준비: Wine 3분할 대신 8 : 2 · CV는 train 안에서만 (원본은 전체 데이터로 CV → test가 선택에 섞임)
import numpy as np                                                                   # 배열
import pandas as pd                                                                  # 표
from sklearn.datasets import load_wine                                               # Wine 178병
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier          # 부스팅 2종
# 회귀 · 히스토그램
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier                                  # 비교용 배깅
from sklearn.metrics import accuracy_score, classification_report                    # 분류 지표
from sklearn.metrics import mean_squared_error, r2_score                             # 회귀 지표
from sklearn.model_selection import GridSearchCV, StratifiedKFold                    # 탐색 · 등분
from sklearn.model_selection import cross_val_score, train_test_split                # CV · 분할
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text  # 트리
from xgboost import XGBClassifier                                                    # XGBoost

SEED = 42                                                          # 원본과 같은 seed
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)  # 오늘의 CV
wine = load_wine()                                                 # 3품종 · 13특성
X, y = wine.data, wine.target                                      # 문제 · 정답
X_train, X_test, y_train, y_test = train_test_split(               # test 36병 봉인
    X, y, stratify=y, test_size=0.2, random_state=SEED)
print("train", X_train.shape, "· test", X_test.shape)              # 142 · 36

train (142, 13) · test (36, 13)


### 원본 목표 · Bagging vs Boosting

- Bagging(Random Forest)이 **분산(Variance)** 을 줄였다면, 이번엔 **편향(Bias)** 을 줄이는 **Boosting**
- Boosting → AdaBoost → Gradient Boosting → XGBoost / LightGBM을 직관 + 코드 + 그림으로

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">구분</th>
<th style="text-align: left;">Bagging</th>
<th style="text-align: left;">Boosting</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">학습 방식 · 모델 간 관계</td>
<td style="text-align: left;">병렬 · 독립</td>
<td style="text-align: left;">순차 · 앞 모델의 잔차(오차)를 반영</td>
</tr>
<tr>
<td style="text-align: left;">목표</td>
<td style="text-align: left;">Variance ↓ (안정성 ↑)</td>
<td style="text-align: left;">Bias ↓ (정확성 ↑) · 단 Variance ↑ 가능</td>
</tr>
<tr>
<td style="text-align: left;">데이터 · 비유</td>
<td style="text-align: left;">Bootstrap · 여러 명이 동시에 팀플</td>
<td style="text-align: left;">잔차 기반 반복 · 오답 노트로 단계적 보완</td>
</tr>
<tr>
<td style="text-align: left;">대표 알고리즘</td>
<td style="text-align: left;">Random Forest</td>
<td style="text-align: left;">AdaBoost · Gradient Boosting · XGBoost · LightGBM · CatBoost</td>
</tr>
</tbody>
</table>

**원본 힌트**

- Boosting 핵심 로직 · 첫 모델 예측 → 잔차(실제 − 예측) → 다음 모델이 잔차를 예측 → 최종 = 이전 예측 + 보정
- 원본 마지막 숙제 Q1 ~ Q5(언제 Bagging · Boosting · 얕은 트리 vs 깊은 트리 · 약한 학습기 · Hessian)는 6장 종합 실습과 면접 답변에서 다룸

---
## 1. 회귀 트리와 잔차

> **이 장의 흐름**
>
> - sin + 노이즈 80개에 회귀 트리(깊이 3) → 구간별 평균의 계단
> - 그다음 평균에서 출발해 깊이 1 트리로 잔차를 5번 보정

<img src="images/pfm10_regtree.png" alt="회귀 트리의 계단 예측과 잔차 보정" width="871">

**파이프라인 위치** · ④ 모델 — 부스팅의 재료

### 1-1. `A` 회귀 트리 · 구간 평균 · `DecisionTreeRegressor` · `export_text` <sub>원본 셀 6 ~ 12</sub>

> 🎯 **회귀 트리는 잎에서 무엇을 내고, 깊이 3이면 예측값이 몇 종류인가?**

💡 **쉽게 말하면**

- 분류 트리 잎 = **다수결 클래스** · 회귀 트리 잎 = **평균값**.
- Gradient Boosting은 잔차(숫자)를 맞추므로 각 스텝이 회귀 트리.

**실행 흐름**

1. sin(x) + 노이즈 80개
2. 깊이 3 회귀 트리
3. R² · RMSE · 잎 · 예측값 종류

**먼저 예측** · 깊이 3이면 서로 다른 예측값은 몇 개? — 답은 코드 실행 결과 아래에

In [2]:
np.random.seed(42)                                                                # 원본 셀 8
X_reg = np.sort(np.random.rand(80, 1) * 10, axis=0)                               # x 0 ~ 10
y_reg = np.sin(X_reg).ravel() + np.random.normal(0, 0.2, 80)                      # sin + 노이즈
reg_tree = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_reg, y_reg)  # 깊이 3
p = reg_tree.predict(X_reg)                                                       # 학습 데이터 예측
print(f"R² Score : {r2_score(y_reg, p):.3f}")                                     # 원본 형식
print(f"RMSE     : {np.sqrt(mean_squared_error(y_reg, p)):.3f}")                  # 원본 형식
print("잎 개수:", reg_tree.get_n_leaves(), "· 서로 다른 예측값:", len(np.unique(p)))  # 계단 수
# 첫 질문 (원본은 graphviz)
print(export_text(reg_tree, feature_names=["x"], decimals=2).splitlines()[0])

R² Score : 0.864
RMSE     : 0.256
잎 개수: 8 · 서로 다른 예측값: 8
|--- x <= 2.65


**결과** · 예측 정답은 8개 — 잎 8개 = 계단 8칸

- `R² Score : 0.864` — 학습 데이터 분산의 86% 설명
- `RMSE     : 0.256` — 노이즈(0.2) 근처까지 맞춤
- `잎 개수: 8 · 서로 다른 예측값: 8` — 깊이 3 = 잎 최대 2³ = 8 · 잎마다 평균 하나
- `|--- x <= 2.65` — 첫 질문 · 이후 구간을 계속 반으로

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">분류 트리</td>
<td style="text-align: left;">잎 = 다수결 · 불순도 Gini / Entropy</td>
<td style="text-align: left;"><code>DecisionTreeClassifier</code></td>
</tr>
<tr>
<td style="text-align: left;">회귀 트리</td>
<td style="text-align: left;">잎 = 평균 · 기준 MSE(squared_error)</td>
<td style="text-align: left;"><code>DecisionTreeRegressor</code></td>
</tr>
<tr>
<td style="text-align: left;">깊이</td>
<td style="text-align: left;">깊을수록 계단이 잘게 · 과적합 위험</td>
<td style="text-align: left;">부스팅에서는 보통 얕게</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 원본의 graphviz 그림은 이 환경에 graphviz가 없어 `export_text` 글자 트리로 대신함
> - 원본 셀 10의 'mse' 표기는 최신 sklearn에서 'squared_error'로 이름이 바뀜

> ✅ **한 줄 정리** · 회귀 트리는 구간마다 평균값 하나를 내는 계단 함수이며, 부스팅은 이 회귀 트리로 잔차를 맞춘다.

**딥러닝 코드에서는**

- 회귀 출력층(Linear 1개) + MSE 손실 — 숫자를 예측하는 같은 문제

### 1-2. `A` 잔차를 순차로 보정 · `resid = y - pred` · `learning_rate` <sub>원본 셀 13 ~ 16</sub>

> 🎯 **평균에서 출발해 깊이 1 트리로 잔차를 5번 보정하면 MSE는 어떻게 되는가?**

💡 **쉽게 말하면**

- **오답 노트** · 틀린 만큼(잔차)을 다음 모델이 맞춤.
- 최종 예측 = 이전 예측 + 학습률 × 보정.

**실행 흐름**

1. 평균으로 첫 예측
2. 잔차 → 깊이 1 트리로 학습
3. 0.5 × 보정을 더함 · 5번 반복

**먼저 예측** · 5번 보정하면 MSE는 평균만 쓴 0.481의 절반쯤일까? — 답은 코드 실행 결과 아래에

In [3]:
pred = np.full_like(y_reg, y_reg.mean())                                # 1. 첫 예측 = 평균
print(f"출발(평균만) MSE {np.mean((y_reg - pred) ** 2):.4f}")           # 기준
for step in range(1, 6):                                                # 순차 5번
    resid = y_reg - pred                                                # 2. 잔차 = 실제 - 예측
    stump = DecisionTreeRegressor(max_depth=1, random_state=42).fit(X_reg, resid)  # 3. 잔차를 학습
    pred = pred + 0.5 * stump.predict(X_reg)                            # 4. 학습률 0.5만큼 보정
    print(f"step {step}: 잔차 MSE {np.mean((y_reg - pred) ** 2):.4f}")  # 줄어드는지

출발(평균만) MSE 0.4810
step 1: 잔차 MSE 0.3934
step 2: 잔차 MSE 0.3457
step 3: 잔차 MSE 0.2997
step 4: 잔차 MSE 0.2718
step 5: 잔차 MSE 0.2441


**결과** · 예측 정답은 절반쯤 — 0.481 → 0.244

- `출발(평균만) MSE 0.4810` — 평균 하나로 예측
- `step 1: 잔차 MSE 0.3934` — 첫 보정
- `step 5: 잔차 MSE 0.2441` — 매 스텝 조금씩 줄어듦 · 약한 모델을 쌓은 효과

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Bagging</td>
<td style="text-align: left;">독립 모델 평균 → 분산 ↓</td>
<td style="text-align: left;">깊은 트리 여러 개</td>
</tr>
<tr>
<td style="text-align: left;">Boosting</td>
<td style="text-align: left;">앞 모델의 잔차를 다음이 보정 → 편향 ↓</td>
<td style="text-align: left;">얕은 트리를 순차로</td>
</tr>
<tr>
<td style="text-align: left;">learning_rate</td>
<td style="text-align: left;">한 스텝 보정 강도(보폭)</td>
<td style="text-align: left;">작을수록 안정 · 스텝이 더 필요</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 보정을 계속 쌓으면 결국 노이즈까지 맞춤 → 스텝 수 · 학습률을 CV로 (2 · 3장)

> ✅ **한 줄 정리** · 부스팅은 잔차를 맞추는 약한 모델을 순차로 더해 오차를 조금씩 줄이는 방법이다.

**딥러닝 코드에서는**

- ResNet의 잔차 연결 · 경사하강의 반복 업데이트와 같은 '조금씩 보정' 생각

---
## 2. AdaBoost

> **이 장의 흐름**
>
> - 깊이 1 트리(stump)를 n_estimators만큼 쌓으며 train · CV 비교
> - 그다음 learning_rate × n_estimators를 train 안 CV로 → 고른 조합으로 test 1회

<img src="images/pfm10_ada.png" alt="AdaBoost · learning_rate × n_estimators CV 히트맵" width="495">

**파이프라인 위치** · ④ 모델 → ⑤ CV 튜닝 → ⑥ test 1회

### 2-1. `A` stump 쌓기 · n_estimators · `AdaBoostClassifier` · `max_depth=1` <sub>원본 셀 17 ~ 26</sub>

> 🎯 **stump 하나는 몇 점이고, 몇 개를 쌓으면 CV가 가장 높은가?**

💡 **쉽게 말하면**

- 처음엔 모든 샘플 가중치가 같음 → **틀린 샘플 가중치 ↑** → 다음 stump가 그 샘플에 집중.
- 모델마다 성능에 따른 가중치 α · 최종 = α 가중 투표.

**실행 흐름**

1. stump 하나의 CV
2. n_estimators 6가지 (learning_rate 0.5)
3. train · train 안 CV

**먼저 예측** · stump 하나의 CV는? 300개면 계속 오를까? — 답은 코드 실행 결과 아래에

In [4]:
stump = DecisionTreeClassifier(max_depth=1, random_state=42)                   # 약한 학습기
print("stump 하나 CV:", round(cross_val_score(stump, X_train, y_train, cv=cv).mean(), 4))  # 출발점
rows = []                                                                      # n마다
for n in [10, 30, 50, 100, 200, 300]:                                          # 원본 셀 25
    ada = AdaBoostClassifier(estimator=stump, n_estimators=n,                  # stump n개
                             learning_rate=0.5, random_state=42)               # 보정 강도 0.5
    s = cross_val_score(ada, X_train, y_train, cv=cv, n_jobs=-1).mean()        # train 안에서만
    tr = ada.fit(X_train, y_train).score(X_train, y_train)                     # 외운 문제
    rows.append([n, tr, s])                                                    # 한 줄
table = pd.DataFrame(rows, columns=["n_estimators", "train", "CV(train 안)"])  # 비교표
print(table.round(4).to_string(index=False))                                   # 소수 넷째 자리

stump 하나 CV: 0.6268


 n_estimators  train  CV(train 안)
           10 0.9789       0.8808
           30 1.0000       0.9512
           50 1.0000       0.9441
          100 1.0000       0.9510
          200 1.0000       0.9441
          300 1.0000       0.9372


**결과** · 예측 정답은 0.6268 · 아님 — 30개 근처에서 최고 후 평평 · 조금 하락

- `stump 하나 CV: 0.6268` — 질문 한 번 → 3품종 중 하나를 못 가름 (약한 학습기)
- `10 0.9789       0.8808` — 10개만 쌓아도 0.63 → 0.88
- `30 1.0000       0.9512` — CV 최고 근처 · train은 이미 1.0
- `300 1.0000       0.9372` — 더 쌓으면 CV가 조금 내려감

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">stump (깊이 1)</td>
<td style="text-align: left;">질문 한 번 · 편향 큼</td>
<td style="text-align: left;">AdaBoost의 기본 약한 학습기</td>
</tr>
<tr>
<td style="text-align: left;">n_estimators</td>
<td style="text-align: left;">약한 모델 개수 · 너무 적으면 과소 · 많으면 과적합 위험</td>
<td style="text-align: left;">CV로</td>
</tr>
<tr>
<td style="text-align: left;">learning_rate</td>
<td style="text-align: left;">한 단계 보정 강도</td>
<td style="text-align: left;">작을수록 안정 · 스텝이 더 필요 (2-2)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 원본 셀 25는 CV를 X, y **전체**로 돌리고 같은 test 점수를 함께 봄 → test가 CV에 섞임 · 여기서는 train 안 CV만
> - 가중치에 집착하면 노이즈 · 이상치에 과적합 (원본 슬라이드 '가중치의 함정')

> ✅ **한 줄 정리** · AdaBoost는 틀린 샘플의 가중치를 올려 약한 stump를 순차로 쌓으며, 몇 개를 쌓을지는 train 안의 CV로 정한다.

**딥러닝 코드에서는**

- 어려운 샘플에 가중치를 주는 focal loss · hard example mining과 같은 생각

### 2-2. `A` learning_rate × n_estimators · `GridSearchCV` · `pivot_table` <sub>원본 셀 27 ~ 29</sub>

> 🎯 **학습률 5개 × 스텝 수 5개 중 CV가 가장 높은 조합은? 작은 학습률은 무엇을 더 필요로 하는가?**

💡 **쉽게 말하면**

- 학습률이 작으면 **천천히** 좋아짐 → 스텝이 더 필요.
- 고원(plateau)처럼 높고 평평한 구간을 고르는 게 합리적.

**실행 흐름**

1. 25조합 × 5-Fold (train 안)
2. 히트맵 표
3. 최고 조합 · 동점 수

**먼저 예측** · 학습률 0.01이면 300스텝으로 충분할까? — 답은 코드 실행 결과 아래에

In [5]:
lr_list, n_list = [0.3, 0.2, 0.1, 0.05, 0.01], [30, 50, 100, 200, 300]          # 원본 셀 28
base = AdaBoostClassifier(estimator=stump, random_state=42)                     # 같은 stump
ada_grid = GridSearchCV(base, {"learning_rate": lr_list, "n_estimators": n_list},  # 25조합
                        cv=cv, n_jobs=-1).fit(X_train, y_train)
res = pd.DataFrame(ada_grid.cv_results_)                                        # 조합별 결과
heat = res.pivot_table(index="param_learning_rate",                             # 행 = 학습률
                       columns="param_n_estimators", values="mean_test_score")  # 열 = 스텝 수
print(heat.sort_index(ascending=False).round(3).to_string())                    # 원본 히트맵의 숫자
print("Best by CV:", ada_grid.best_params_, round(ada_grid.best_score_, 3))     # 최고
print("공동 1위:", int((res["rank_test_score"] == 1).sum()), "개")              # 동점 확인

param_n_estimators     30     50     100    200    300
param_learning_rate                                   
0.30                 0.930  0.923  0.937  0.951  0.951
0.20                 0.888  0.916  0.937  0.937  0.944
0.10                 0.846  0.867  0.916  0.930  0.937
0.05                 0.831  0.838  0.860  0.923  0.930
0.01                 0.627  0.704  0.817  0.846  0.838
Best by CV: {'learning_rate': 0.3, 'n_estimators': 200} 0.951
공동 1위: 2 개


**결과** · 예측 정답은 아님 — 0.01은 300스텝에도 0.838

- `0.30                 0.930  0.923  0.937  0.951  0.951` — 학습률이 크면 빨리 높아짐
- `0.01                 0.627  0.704  0.817  0.846  0.838` — 학습률이 작으면 300스텝으로도 부족
- `Best by CV: {'learning_rate': 0.3, 'n_estimators': 200} 0.951` — 최고 조합
- `공동 1위: 2 개` — 0.3 × 200 · 0.3 × 300이 같은 점수

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">학습률 큼 (0.3)</td>
<td style="text-align: left;">빨리 좋아짐 · 과적합 위험 ↑</td>
<td style="text-align: left;">스텝이 적어도 됨</td>
</tr>
<tr>
<td style="text-align: left;">학습률 작음 (0.01)</td>
<td style="text-align: left;">천천히 · 안정적</td>
<td style="text-align: left;">스텝을 훨씬 많이</td>
</tr>
<tr>
<td style="text-align: left;">고원 고르기</td>
<td style="text-align: left;">높고 평평한 구간</td>
<td style="text-align: left;">1등 한 칸보다 안정적</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 원본(전체 데이터 CV)의 최고는 0.3 × 100 · 0.944 → train 안 CV로 바꾸면 0.3 × 200 · 0.951
> - 작은 학습률 + 적은 스텝 = 과소적합 (학습률 0.01 · 30스텝 → 0.627 ≈ stump 하나)

> ✅ **한 줄 정리** · 학습률과 스텝 수는 짝으로 움직이므로 함께 탐색하고, train 안의 CV에서 높고 평평한 구간을 고른다.

**딥러닝 코드에서는**

- 학습률 × epoch 수도 같은 짝 · 작은 학습률이면 더 오래 학습

### 2-3. `B` 고른 조합으로 test 1회 · `best_estimator_` · `classification_report` <sub>원본 셀 30 ~ 33</sub>

> 🎯 **CV로 고른 AdaBoost는 test 36병에서 몇 점인가?**

💡 **쉽게 말하면**

- 선택은 이미 끝남 → test는 **한 번만**.
- 클래스별 표로 어떤 품종을 틀렸는지 확인.

**실행 흐름**

1. best_estimator_ (train 전체로 재학습됨)
2. test 예측
3. 정확도 · 클래스별 표

**먼저 예측** · test 정확도는 CV 0.951보다 높을까? — 답은 코드 실행 결과 아래에

In [6]:
y_pred = ada_grid.best_estimator_.predict(X_test)                              # test 첫 사용
print(f"[AdaBoost Best] Test Accuracy: {accuracy_score(y_test, y_pred):.3f}")  # 원본 형식
print(classification_report(y_test, y_pred, target_names=wine.target_names, digits=3))  # 클래스별

[AdaBoost Best] Test Accuracy: 0.972
              precision    recall  f1-score   support

     class_0      0.923     1.000     0.960        12
     class_1      1.000     0.929     0.963        14
     class_2      1.000     1.000     1.000        10

    accuracy                          0.972        36
   macro avg      0.974     0.976     0.974        36
weighted avg      0.974     0.972     0.972        36



**결과** · 예측 정답은 높음 — 0.972 (36병 중 35병)

- `[AdaBoost Best] Test Accuracy: 0.972` — 원본과 같은 test 점수
- `class_1      1.000     0.929     0.963        14` — class_1 한 병을 class_0으로

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">CV 0.951</td>
<td style="text-align: left;">고를 때 쓴 점수</td>
<td style="text-align: left;">선택</td>
</tr>
<tr>
<td style="text-align: left;">test 0.972</td>
<td style="text-align: left;">처음 보는 36병</td>
<td style="text-align: left;">보고</td>
</tr>
<tr>
<td style="text-align: left;">과적합 신호</td>
<td style="text-align: left;">test와 CV 간극 ↑ · 특정 클래스 오분류 급증</td>
<td style="text-align: left;">둘 다 볼 것</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 불균형 문제라면 정확도 말고 F1 · ROC-AUC · PR-AP도 함께 (파일별 4)

> ✅ **한 줄 정리** · train 안의 CV로 고른 조합으로 test를 한 번 평가하고, 클래스별 표로 어디서 틀렸는지 확인한다.

**딥러닝 코드에서는**

- val로 고른 설정 → test 1회 · 클래스별 정밀도 · 재현율까지 보고

---
## 3. Gradient Boosting

> **이 장의 흐름**
>
> - f(x) = x²에서 학습률별 경사하강 → 너무 작음 · 적절 · 너무 큼
> - 그다음 GBM 64조합을 train 안 CV로 → 고른 조합으로 test 1회

<img src="images/pfm10_gd.png" alt="f(x) = x² 경사하강 · 학습률별 경로" width="678">

**파이프라인 위치** · ④ 모델 → ⑤ CV 튜닝 → ⑥ test 1회

### 3-1. `A` 경사하강 · 학습률 · `x -= lr * 2 * x` <sub>원본 셀 43 ~ 46</sub>

> 🎯 **f(x) = x²를 x = 5에서 내려갈 때 학습률 0.01 · 0.1 · 0.3 · 1.1은 각각 어디에 도착하는가?**

💡 **쉽게 말하면**

- 기울기 = Loss가 얼마나 가파른지 알려 주는 **나침반** · 반대 방향으로 조금씩.
- Gradient Boosting은 잔차 = 손실의 기울기 방향을 따라 트리를 더함.

**실행 흐름**

1. f(x) = x² · 기울기 2x
2. 학습률 4가지로 반복
3. 도착 위치 · loss

**먼저 예측** · 학습률 1.1이면? — 답은 코드 실행 결과 아래에

In [7]:
for lr, steps in [(0.01, 80), (0.1, 20), (0.3, 10), (1.1, 10)]:      # 원본 3개 + 발산 예시
    x = 5.0                                                          # 출발점
    for _ in range(steps):                                           # 걸음마다
        x -= lr * 2 * x                                              # 기울기(2x)의 반대 방향으로
    print(f"lr={lr}: {steps}걸음 뒤 x={x:.4f} · loss={x ** 2:.4f}")  # 도착

lr=0.01: 80걸음 뒤 x=0.9932 · loss=0.9865
lr=0.1: 20걸음 뒤 x=0.0576 · loss=0.0033
lr=0.3: 10걸음 뒤 x=0.0005 · loss=0.0000
lr=1.1: 10걸음 뒤 x=30.9587 · loss=958.4400


**결과** · 예측 정답은 발산 — loss 958

- `lr=0.01: 80걸음 뒤 x=0.9932 · loss=0.9865` — 너무 작음 · 80걸음에도 아직 멂
- `lr=0.1: 20걸음 뒤 x=0.0576 · loss=0.0033` — 적절 · 안정적으로 도착
- `lr=0.3: 10걸음 뒤 x=0.0005 · loss=0.0000` — 이 함수에서는 0.3도 빠르게 수렴
- `lr=1.1: 10걸음 뒤 x=30.9587 · loss=958.4400` — 너무 큼 · 최소점을 지나쳐 발산

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">너무 작을 때</td>
<td style="text-align: left;">천천히 · 시간이 오래</td>
<td style="text-align: left;">느림</td>
</tr>
<tr>
<td style="text-align: left;">적절할 때</td>
<td style="text-align: left;">안정적으로 최소점</td>
<td style="text-align: left;">이상적</td>
</tr>
<tr>
<td style="text-align: left;">너무 클 때</td>
<td style="text-align: left;">보폭이 커서 지나치며 발산</td>
<td style="text-align: left;">불안정</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 원본은 0.3을 '너무 큼'으로 분류했지만 f(x) = x²에서는 1보다 작으면 수렴 → 발산은 1을 넘을 때 (1.1로 추가 확인)

> ✅ **한 줄 정리** · 경사하강은 기울기의 반대 방향으로 학습률만큼 이동하며, 학습률이 너무 작으면 느리고 너무 크면 발산한다.

**딥러닝 코드에서는**

- 신경망 학습 = 같은 경사하강 · 학습률이 가장 중요한 하이퍼파라미터

### 3-2. `A` GBM GridSearch (train 안) · `GradientBoostingClassifier` · `subsample` <sub>원본 셀 47 ~ 49</sub>

> 🎯 **학습률 · 스텝 · subsample · 깊이 64조합을 train 안 CV로 고르면?**

💡 **쉽게 말하면**

- AdaBoost = **틀린 샘플 가중치 ↑** · GBM = **손실의 기울기(잔차)** 를 따라감.
- 그래서 GBM은 손실 함수를 바꿔 회귀 · 다중 분류 · 사용자 손실까지.

**실행 흐름**

1. 64조합 × 5-Fold (train 142)
2. 최고 조합 · 점수
3. 공동 1위 수

**먼저 예측** · 원본(전체 데이터)의 최고 lr 0.3과 같을까? — 답은 코드 실행 결과 아래에

In [8]:
param_grid = {"learning_rate": [0.3, 0.1, 0.05, 0.01],                      # 원본 셀 48
              "n_estimators": [50, 100, 200, 300],                          # 스텝 수
              "subsample": [1.0, 0.8], "max_depth": [3, 2]}                 # 4 · 4 · 2 · 2 = 64조합
gs = GridSearchCV(GradientBoostingClassifier(random_state=42), param_grid,  # 원본은 X, y 전체로 fit
                  cv=cv, n_jobs=-1).fit(X_train, y_train)
print("best_params_ :", gs.best_params_)                                    # 고른 조합
print("cv best_score:", round(gs.best_score_, 4))                           # 고를 때 쓴 점수
print("공동 1위:", int((gs.cv_results_["rank_test_score"] == 1).sum()), "/ 64")  # 동점

best_params_ : {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 100, 'subsample': 0.8}
cv best_score: 0.9719
공동 1위: 1 / 64


**결과** · 예측 정답은 아님 — lr 0.05 · 깊이 2 · 100 · subsample 0.8

- `best_params_ : {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 100, 'subsample': 0.8}` — 얕은 트리(깊이 2) · 행 80%씩
- `cv best_score: 0.9719` — train 안 CV
- `공동 1위: 1 / 64` — 이번엔 유일한 1위

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">AdaBoost</td>
<td style="text-align: left;">틀린 샘플 가중치 ↑ · 분류 중심</td>
<td style="text-align: left;"><code>AdaBoostClassifier</code></td>
</tr>
<tr>
<td style="text-align: left;">Gradient Boosting</td>
<td style="text-align: left;">손실의 기울기 ↓ · 회귀 · 분류 모두</td>
<td style="text-align: left;"><code>GradientBoostingClassifier</code></td>
</tr>
<tr>
<td style="text-align: left;">subsample &lt; 1</td>
<td style="text-align: left;">스텝마다 행 일부만 → 분산 ↓ · OOB 추정 가능</td>
<td style="text-align: left;">stochastic GBM</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 원본 셀 48은 GridSearch를 X, y **전체**로 fit하고 셀 50에서 그 안의 test로 평가 → test가 선택에 쓰임
> - 그래서 원본 결과(lr 0.3 · CV 0.9773)와 이 노트북 결과가 다름

> ✅ **한 줄 정리** · Gradient Boosting은 손실의 기울기를 따라 트리를 더하며, 학습률 · 스텝 · 깊이 · subsample은 train 안의 CV로 함께 고른다.

**딥러닝 코드에서는**

- 모든 하이퍼파라미터 탐색은 val(또는 내부 CV)에서만

### 3-3. `B` 고른 GBM으로 test 1회 · `best_estimator_` <sub>원본 셀 50</sub>

> 🎯 **train 안 CV로 고른 GBM의 test 정확도는?**

💡 **쉽게 말하면**

- 선택은 train 안에서 끝남 → test는 이번이 처음이자 마지막.
- 원본과 달리 test가 선택에 섞이지 않음.

**실행 흐름**

1. best_estimator_
2. test 예측
3. 정확도

**먼저 예측** · CV 0.9719보다 높을까? — 답은 코드 실행 결과 아래에

In [9]:
y_pred_gb = gs.best_estimator_.predict(X_test)                          # test 첫 사용
print(f"Final Test Accuracy: {accuracy_score(y_test, y_pred_gb):.3f}")  # 원본 형식
print("틀린 병:", int((y_pred_gb != y_test).sum()), "/ 36")             # 1병 = 0.028

Final Test Accuracy: 1.000
틀린 병: 0 / 36


**결과** · 예측 정답은 높음 — 1.000

- `Final Test Accuracy: 1.000` — 36병 전부 정답 (원본은 0.972)
- `틀린 병: 0 / 36` — 36병에서 1병 = 0.028 → 운도 섞임

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">원본 절차</td>
<td style="text-align: left;">전체로 탐색 → 그 안의 test로 평가</td>
<td style="text-align: left;">test가 선택에 섞임</td>
</tr>
<tr>
<td style="text-align: left;">이 노트북</td>
<td style="text-align: left;">train 안에서 탐색 → test 1회</td>
<td style="text-align: left;">공정한 평가</td>
</tr>
<tr>
<td style="text-align: left;">결과 차이</td>
<td style="text-align: left;">0.972 vs 1.000</td>
<td style="text-align: left;">작은 test에서는 흔들림이 큼</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 1.000이라고 '완벽한 모델'이 아님 · 36병에서 1병 = 0.028

> ✅ **한 줄 정리** · test가 선택에 섞이지 않게 train 안에서 고른 뒤 test를 한 번 평가해야 그 점수를 보고할 수 있다.

**딥러닝 코드에서는**

- test 세트로 early stopping · 튜닝을 하면 같은 누수

---
## 4. 손실 함수

> **이 장의 흐름**
>
> - 분류 · log_loss(기본)의 CV와 exponential(이진 전용) 확인
> - 그다음 sin + 이상치 2개 회귀에서 squared · absolute · huber를 '진짜 sin과의 거리'로 비교

<img src="images/pfm10_loss.png" alt="이상치 2개 · 손실 함수별 예측선" width="711">

**파이프라인 위치** · ④ 모델 — 문제 정의(손실)

### 4-1. `B` 분류 손실 · log_loss · `loss="log_loss"` · `loss="exponential"` <sub>원본 셀 51 ~ 54</sub>

> 🎯 **Wine(3품종)에 log_loss GBM의 CV는? exponential을 쓰면?**

💡 **쉽게 말하면**

- log_loss = 교차 엔트로피 (확률 출력) · 분류의 기본.
- exponential = AdaBoost와 같은 손실 · **이진 분류 전용**.

**실행 흐름**

1. log_loss GBM CV (train 안)
2. exponential로 fit 시도

**먼저 예측** · 3품종에 exponential을 쓰면? — 답은 코드 실행 결과 아래에

In [10]:
gb_loss = GradientBoostingClassifier(                                     # 원본 셀 54
    loss="log_loss", n_estimators=200, learning_rate=0.1,                 # 분류 기본 손실
    max_depth=2, subsample=0.8, random_state=42)                          # 얕은 트리 · 행 80%
s = cross_val_score(gb_loss, X_train, y_train, cv=cv)                     # train 안
print(f"[분류] loss='log_loss' · CV Accuracy: {s.mean():.3f} ± {s.std():.3f}")  # 원본 형식
try:                                                                      # 원본은 주석으로만 경고
    GradientBoostingClassifier(loss="exponential").fit(X_train, y_train)  # 3클래스
except ValueError as e:                                                   # sklearn이 막음
    print("exponential:", str(e)[:70])                                    # 이진 전용

[분류] loss='log_loss' · CV Accuracy: 0.958 ± 0.034
exponential: loss='exponential' is only suitable for a binary classification proble


**결과** · 예측 정답은 ValueError — 이진 분류 전용

- `[분류] loss='log_loss' · CV Accuracy: 0.958 ± 0.034` — train 안 CV (원본은 전체 데이터로 0.977)
- `exponential: loss='exponential' is only suitable for a binary classification proble` — 3클래스에는 못 씀

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>log_loss</code> (기본)</td>
<td style="text-align: left;">교차 엔트로피 · 확률 출력</td>
<td style="text-align: left;">대부분의 분류</td>
</tr>
<tr>
<td style="text-align: left;"><code>exponential</code></td>
<td style="text-align: left;">AdaBoost와 같은 지수 손실</td>
<td style="text-align: left;">이진 분류에서만</td>
</tr>
<tr>
<td style="text-align: left;">손실 바꾸기</td>
<td style="text-align: left;">= 문제를 다시 정의</td>
<td style="text-align: left;">데이터 · 목표에 맞게</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 원본 셀 54의 CV는 X, y 전체 → 같은 계약으로 비교하려면 train 안 CV

> ✅ **한 줄 정리** · 분류 GBM의 기본 손실은 log_loss이며, exponential은 이진 분류에서만 쓸 수 있다.

**딥러닝 코드에서는**

- `CrossEntropyLoss`(다중) · `BCEWithLogitsLoss`(이진) — 손실 선택 = 출력 형태 선택

### 4-2. `A` 이상치와 회귀 손실 · `squared_error` · `absolute_error` · `huber` <sub>원본 셀 55 ~ 59</sub>

> 🎯 **이상치 2개가 섞인 sin 데이터에서 어떤 손실이 진짜 sin에 가장 가까운가?**

💡 **쉽게 말하면**

- squared_error = 큰 오차를 **제곱**으로 벌줌 → 이상치를 맞추려 끌려감.
- absolute_error = 절댓값 → 이상치에 덜 끌림 · huber = 둘의 혼합.

**실행 흐름**

1. y[10] = 5 · y[50] = -4 이상치
2. 손실 3가지로 GBM 회귀
3. 데이터 RMSE vs 진짜 sin RMSE

**먼저 예측** · 이상치 포함 데이터 RMSE가 가장 낮은 손실이 가장 좋은 모델일까? — 답은 코드 실행 결과 아래에

In [11]:
y_out = y_reg.copy()                                                   # 원본 셀 56
y_out[10], y_out[50] = 5.0, -4.0                                       # 이상치 2개
X_plot = np.linspace(0, 10, 500).reshape(-1, 1)                        # 진짜 sin과 비교할 x
for loss in ["squared_error", "absolute_error", "huber"]:              # 원본 셀 58
    gb = GradientBoostingRegressor(loss=loss, n_estimators=200, learning_rate=0.1,  # 같은 설정
                                   max_depth=3, random_state=42).fit(X_reg, y_out)
    rmse_data = np.sqrt(mean_squared_error(y_out, gb.predict(X_reg)))  # 이상치 포함 데이터와의 거리
    # 진짜 함수와의 거리
    rmse_true = np.sqrt(mean_squared_error(np.sin(X_plot).ravel(), gb.predict(X_plot)))
    print(f"{loss:<15s} RMSE(이상치 포함 데이터) {rmse_data:.3f}",     # 학습 데이터 기준
          f"· RMSE(진짜 sin) {rmse_true:.3f}")                         # 진짜 함수 기준

squared_error   RMSE(이상치 포함 데이터) 0.039 · RMSE(진짜 sin) 0.542


absolute_error  RMSE(이상치 포함 데이터) 0.637 · RMSE(진짜 sin) 0.119


huber           RMSE(이상치 포함 데이터) 0.136 · RMSE(진짜 sin) 0.530


**결과** · 예측 정답은 아님 — 데이터 RMSE 최저인 squared_error가 진짜 sin과는 가장 멂

- `squared_error   RMSE(이상치 포함 데이터) 0.039 · RMSE(진짜 sin) 0.542` — 이상치까지 맞춰서 데이터 RMSE만 낮음
- `absolute_error  RMSE(이상치 포함 데이터) 0.637 · RMSE(진짜 sin) 0.119` — 이상치를 무시 → 진짜 sin에 가장 가까움
- `huber           RMSE(이상치 포함 데이터) 0.136 · RMSE(진짜 sin) 0.530` — 기본 설정(alpha 0.9)에서는 이상치에 꽤 끌림

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>squared_error</code> (L2 · 기본)</td>
<td style="text-align: left;">큰 오차를 제곱 → 이상치에 민감</td>
<td style="text-align: left;">이상치가 없을 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>absolute_error</code> (L1)</td>
<td style="text-align: left;">중앙값을 예측 → 이상치에 강함</td>
<td style="text-align: left;">이상치가 있을 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>huber</code> · <code>quantile</code></td>
<td style="text-align: left;">L1 + L2 혼합 · 분위수</td>
<td style="text-align: left;">적당히 강건 · 상위 90% 예측 등</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 원본 셀 58의 RMSE 표(0.039 · 0.637 · 0.136)는 **이상치 포함 학습 데이터**와의 거리라 낮을수록 이상치를 쫓은 것 → '가장 좋다'로 읽으면 반대
> - 판단은 진짜 함수(또는 깨끗한 검증 데이터)와의 거리로 · 이상치 대응은 스케일러 선택과 같은 흐름 (파일별 3)

> ✅ **한 줄 정리** · 이상치가 있으면 squared_error는 이상치를 쫓아 학습 RMSE만 낮아지므로, 손실 선택은 깨끗한 기준과의 거리로 판단한다.

**딥러닝 코드에서는**

- `nn.HuberLoss` · `nn.L1Loss` — 라벨 노이즈가 있는 회귀에서 MSE 대신

---
## 5. XGBoost · LightGBM · CatBoost

> **이 장의 흐름**
>
> - LightGBM의 히스토그램 bin을 sklearn HistGradientBoosting의 max_bins로 실험
> - 그다음 범주형 열을 원-핫 + XGBoost vs 네이티브 범주 처리 → 부스팅 3종 CV 비교

<img src="images/pfm10_bins.png" alt="max_bins별 train 안 CV" width="600">

**파이프라인 위치** · ④ 모델 — 라이브러리 선택

### 5-1. `B` 히스토그램 bin 수 · `HistGradientBoostingClassifier` · `max_bins` <sub>원본 셀 60 ~ 69</sub>

> 🎯 **bin 수를 255 → 10으로 줄이면 작은 데이터(Wine 142병)에서 CV는 떨어지는가?**

💡 **쉽게 말하면**

- 연속값을 **구간(bin)** 으로 묶어 구간 단위로 계산 → 계산량 ↓.
- bin을 줄이면 표현력 ↓ 대신 작은 데이터에서는 **규제 효과**가 생길 수 있음.

**실행 흐름**

1. max_bins 5가지 (LightGBM max_bin과 같은 생각)
2. train 안 CV

**먼저 예측** · bin 10이 가장 낮을까? — 답은 코드 실행 결과 아래에

In [12]:
rows = []                                                                     # bin마다
for b in [10, 31, 63, 127, 255]:                                              # 원본 셀 66의 후보
    # 히스토그램 부스팅
    hgb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, max_bins=b,
                                         max_leaf_nodes=31, random_state=42)  # 잎 31 (leaf-wise)
    rows.append([b, cross_val_score(hgb, X_train, y_train, cv=cv).mean()])    # train 안 CV
# 표
print(pd.DataFrame(rows, columns=["max_bins", "CV(train 안)"]).round(4).to_string(index=False))

 max_bins  CV(train 안)
       10       0.9717
       31       0.9512
       63       0.9650
      127       0.9581
      255       0.9581


**결과** · 예측 정답은 아님 — bin 10이 가장 높음 (0.9717)

- `10       0.9717` — 작은 데이터에서는 bin을 줄여도 손해가 없고 오히려 높음
- `255       0.9581` — bin이 많아도 이득 없음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">데이터 많을 때 (10만 개)</td>
<td style="text-align: left;">bin마다 데이터 충분 · bin 수 영향 작음</td>
<td style="text-align: left;">bin 축소 = 속도 이득만</td>
</tr>
<tr>
<td style="text-align: left;">데이터 적을 때 (수백 개)</td>
<td style="text-align: left;">bin 255면 bin마다 1 ~ 2개 → 불안정</td>
<td style="text-align: left;">bin 축소 = 규제 효과 가능</td>
</tr>
<tr>
<td style="text-align: left;">LightGBM 장점</td>
<td style="text-align: left;">히스토그램 + leaf-wise 성장</td>
<td style="text-align: left;">대용량에서 단점은 희석 · 속도는 극대화</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - lightgbm이 이 환경에 없어 같은 히스토그램 방식인 sklearn `HistGradientBoostingClassifier`로 실험 (원본 LGBM 결과: bin 10 · 31 → 0.961 · 0.960, 63 이상 → 0.966)
> - 원본 결과는 전체 데이터 CV · 이 표는 train 안 CV → 숫자가 다름

> ✅ **한 줄 정리** · 히스토그램 bin 수는 표현력과 안정성의 교환이며, 작은 데이터에서는 bin을 줄여도 손해가 없거나 규제 효과가 생길 수 있다.

**딥러닝 코드에서는**

- 양자화(quantization) · 혼합 정밀도 — 정밀도를 줄여 속도를 얻는 같은 교환

### 5-2. `B` 범주형 열 · 원-핫 vs 네이티브 · `OneHotEncoder` · `categorical_features="from_dtype"` <sub>원본 셀 77 ~ 85</sub>

> 🎯 **범주형 열 2개가 있는 데이터에서 원-핫 + XGBoost와 네이티브 범주 처리 부스팅의 정확도는?**

💡 **쉽게 말하면**

- 기존 방식 · OneHotEncoder + ColumnTransformer + Pipeline (파일별 3).
- CatBoost · `cat_features`만 지정 → 인코딩 없이 (내부 Ordered Target Encoding).

**실행 흐름**

1. 색 · 크기 범주 + 무게 · 가격 200행
2. 원-핫 + XGBoost Pipeline
3. category 타입 + 네이티브 범주 처리

**먼저 예측** · 두 방식의 정확도는 같을까? — 답은 코드 실행 결과 아래에

In [13]:
from sklearn.compose import ColumnTransformer                    # 열마다 다른 전처리
from sklearn.pipeline import Pipeline                            # 전처리 + 모델
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # 원-핫 · 스케일링

np.random.seed(42)                                                           # 원본 셀 80
n = 200                                                                      # 행 수
df = pd.DataFrame({"color": np.random.choice(["red", "green", "blue"], n),   # 범주형
                   "size": np.random.choice(["small", "medium", "large"], n),  # 범주형
                   "weight": np.random.normal(50, 10, n).round(1),           # 수치형
                   "price": np.random.normal(1000, 300, n).round(0)})        # 수치형
df["target"] = ((df["color"] == "red") & (df["weight"] > 50)).astype(int)    # 빨강 + 무거움 → 1
noise_idx = np.random.choice(n, size=15, replace=False)                      # 노이즈 15개
df.loc[noise_idx, "target"] = 1 - df.loc[noise_idx, "target"]                # 뒤집기
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(df.drop(columns="target"), df["target"],  # 8 : 2
                                              test_size=0.2, stratify=df["target"], random_state=42)
pre = ColumnTransformer([("num", StandardScaler(), ["weight", "price"]),     # 원본 셀 82
                         ("cat", OneHotEncoder(handle_unknown="ignore"), ["color", "size"])])
xgb_c = XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=3,      # 원본 셀 83
                      eval_metric="logloss", random_state=42)                # 이진 손실
pipe = Pipeline([("pre", pre), ("model", xgb_c)]).fit(Xc_tr, yc_tr)          # 원-핫 → XGBoost
acc_oh = accuracy_score(yc_te, pipe.predict(Xc_te))                          # 원-핫 방식 정확도
cat_t = {"color": "category", "size": "category"}  # 범주 타입 지정 (cat_features에 해당)
# 인코딩 없이
hgb_c = HistGradientBoostingClassifier(categorical_features="from_dtype", max_iter=200,
                                       learning_rate=0.1, max_depth=3, random_state=42)
hgb_c.fit(Xc_tr.astype(cat_t), yc_tr)                                        # 범주 열을 그대로
acc_nat = accuracy_score(yc_te, hgb_c.predict(Xc_te.astype(cat_t)))          # 네이티브 방식 정확도
print("데이터:", df.shape, "· 타깃", df["target"].value_counts().to_dict())  # 200 · 157 : 43
print(f"원-핫 + XGBoost Accuracy: {acc_oh:.3f}",                             # 방식 1
      f"· 네이티브 범주(HistGB) Accuracy: {acc_nat:.3f}")                    # 방식 2

데이터: (200, 5) · 타깃 {0: 157, 1: 43}
원-핫 + XGBoost Accuracy: 0.950 · 네이티브 범주(HistGB) Accuracy: 0.900


**결과** · 예측 정답은 다름 — 원-핫 + XGB 0.950 · 네이티브 0.900

- `데이터: (200, 5) · 타깃 {0: 157, 1: 43}` — 원본과 같은 데이터
- `원-핫 + XGBoost Accuracy: 0.950 · 네이티브 범주(HistGB) Accuracy: 0.900` — test 40개에서 2개 차이 · 원본 CatBoost는 0.950

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">원-핫 + Pipeline</td>
<td style="text-align: left;">차원 증가 · 모든 모델에 쓸 수 있음</td>
<td style="text-align: left;">XGBoost · LightGBM · 로지스틱</td>
</tr>
<tr>
<td style="text-align: left;">CatBoost <code>cat_features</code></td>
<td style="text-align: left;">인코딩 없이 · Ordered Target Encoding으로 누수 방지</td>
<td style="text-align: left;">범주형이 많은 데이터</td>
</tr>
<tr>
<td style="text-align: left;">HistGB <code>categorical_features</code></td>
<td style="text-align: left;">sklearn의 네이티브 범주 분할</td>
<td style="text-align: left;">catboost 없이 같은 생각</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - catboost가 이 환경에 없어 같은 아이디어(인코딩 없는 범주 처리)를 sklearn HistGradientBoosting으로 실행 → CatBoost와 내부 방식이 다름
> - test 40개에서 2개 차이 → 우열을 정하지 않기 · 원본 결과는 XGB · CatBoost 모두 0.950

> ✅ **한 줄 정리** · 범주형 열은 원-핫으로 펼치거나 네이티브 범주 처리를 쓸 수 있고, 어느 쪽이 나은지는 같은 데이터에서 비교해 정한다.

**딥러닝 코드에서는**

- 범주형 입력 = `nn.Embedding` · 원-핫보다 차원이 작고 학습됨

### 5-3. `B` 부스팅 3종 비교 · `XGBClassifier` · `HistGradientBoostingClassifier` <sub>원본 셀 70 ~ 76 · 86 ~ 88</sub>

> 🎯 **GBM · XGBoost · 히스토그램 부스팅을 같은 train 안 CV로 비교하면?**

💡 **쉽게 말하면**

- XGBoost = GBM + L1 / L2 규제 + 병렬 + **2차 미분(Hessian)**.
- LightGBM = 히스토그램 + leaf-wise · CatBoost = 범주형 자동 처리.

**실행 흐름**

1. 세 모델 같은 설정 (200스텝 · 학습률 0.1)
2. train 안 CV 평균 · 표준편차

**먼저 예측** · 가장 단순한 GBM이 가장 낮을까? — 답은 코드 실행 결과 아래에

In [14]:
gbm = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,  # sklearn 기본 부스팅
                                 max_depth=3, random_state=42)  # 깊이 3
models = {"GBM": gbm,                                           # 세 모델 같은 200스텝 · 학습률 0.1
          "XGBoost": XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=3,  # 원본 셀 87
                                   eval_metric="mlogloss", random_state=42),
          "HistGB": HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1,  # LightGBM 방식
                                                   max_leaf_nodes=31, random_state=42)}
for name, m in models.items():                                  # 같은 계약
    s = cross_val_score(m, X_train, y_train, cv=cv, n_jobs=-1)  # train 안
    print(f"{name:<8s} CV {s.mean():.3f} ± {s.std():.3f}")      # 평균 · 흔들림

GBM      CV 0.923 ± 0.070


XGBoost  CV 0.958 ± 0.051
HistGB   CV 0.958 ± 0.034


**결과** · 예측 정답은 그렇다 — GBM 0.923 · XGB 0.958 · HistGB 0.958

- `GBM      CV 0.923 ± 0.070` — 가장 낮고 흔들림도 가장 큼
- `XGBoost  CV 0.958 ± 0.051` — 규제 덕분에 조금 더 안정
- `HistGB   CV 0.958 ± 0.034` — 히스토그램 방식 · 흔들림 가장 작음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">GBM (sklearn)</td>
<td style="text-align: left;">순차 트리 · depth-wise</td>
<td style="text-align: left;">작은 데이터 · 기준선</td>
</tr>
<tr>
<td style="text-align: left;">XGBoost</td>
<td style="text-align: left;">L1 / L2 규제 · 병렬 · Hessian</td>
<td style="text-align: left;">가장 범용적</td>
</tr>
<tr>
<td style="text-align: left;">LightGBM · CatBoost</td>
<td style="text-align: left;">히스토그램 + leaf-wise · 범주형 자동 + Ordered Boosting</td>
<td style="text-align: left;">대용량 · 범주형이 많을 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - - 원본 표의 'GBM은 규제 X
> - 모델 복잡도 제어 없음'은 지나친 표현
> - sklearn GBM도 max_depth · subsample
> - min_samples_leaf
> - n_iter_no_change(early stopping)로 제어
> - - 원본 셀 76의 `use_label_encoder=False`는 XGBoost 2.0 이후 없어진 옵션
> - 원본 4종 결과(전체 데이터 CV): GBM 0.950 · XGB 0.961 · LGBM 0.966
> - CatBoost 0.972

> ✅ **한 줄 정리** · 부스팅 라이브러리는 규제 · 히스토그램 · 범주 처리 방식이 다르며, 무엇이 나은지는 같은 계약의 CV로 비교한다.

**딥러닝 코드에서는**

- 표 데이터는 부스팅이 강한 기준선 · 이미지 · 텍스트는 신경망

---
## 6. 종합 실습 · 숙제 실험 · 얕은 트리 vs 깊은 트리

> **이 장의 흐름**
>
> - 1 ~ 5장을 한 번에 사용
> - 같은 train 안 CV로 GBM과 RF의 트리 깊이를 1 · 3 · 끝까지로 바꿔 비교

<img src="images/pfm10_practice.png" alt="트리 깊이별 GBM vs RF · train 안 CV" width="595">

**파이프라인 위치** · ⑤ 교차검증 — 앙상블 설계

> 🎯 **깊이를 키우면 GBM과 RF의 CV는 각각 어떻게 되는가?**

💡 **쉽게 말하면**

- Boosting = 약한 모델을 순차로 · **깊은 트리는 첫 스텝에서 다 외워 잔차가 노이즈만**.
- RF = 깊은 트리 여러 개의 평균 · **분산은 평균이 줄여 줌**.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">단계</th>
<th style="text-align: left;">하는 일</th>
<th style="text-align: left;">쓰는 문법</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">깊이 1 · 3 · 끝까지(GBM은 10)</td>
<td style="text-align: left;">1-1 · 파일별 6</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">GBM 200스텝 · train 안 CV · train 정확도</td>
<td style="text-align: left;">3장</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">RF 300그루 · train 안 CV</td>
<td style="text-align: left;">파일별 7</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">Q1 ~ Q5에 근거로 쓰기</td>
<td style="text-align: left;">면접 답변</td>
</tr>
</tbody>
</table>

**먼저 예측** · 깊이를 키우면 GBM은 오를까? RF는? — 답은 코드 실행 결과 아래에

In [15]:
# 깊이만 바꿔 GBM vs RF (train 안 CV · test는 안 씀)
rows = []                                                                         # 깊이마다
for d in [1, 3, None]:                                                            # 1 · 3 · 끝까지
    gbd = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,         # 순차 · 잔차 보정
                                     # GBM은 None 대신 10
                                     max_depth=d if d else 10, random_state=42)
    rfd = RandomForestClassifier(n_estimators=300, max_depth=d, random_state=42)  # 병렬 · 평균
    gb_cv = cross_val_score(gbd, X_train, y_train, cv=cv, n_jobs=-1).mean()       # GBM CV
    gb_tr = gbd.fit(X_train, y_train).score(X_train, y_train)                     # GBM 외운 문제
    rf_cv = cross_val_score(rfd, X_train, y_train, cv=cv, n_jobs=-1).mean()       # RF CV
    rows.append([str(d), gb_cv, gb_tr, rf_cv])                                    # 한 줄
table = pd.DataFrame(rows, columns=["depth", "GB CV", "GB train", "RF CV"])       # 비교표
print(table.round(4).to_string(index=False))                                      # 소수 넷째 자리

depth  GB CV  GB train  RF CV
    1 0.9650       1.0 0.9653
    3 0.9232       1.0 0.9722
 None 0.9020       1.0 0.9791


**결과** · 예측 정답은 반대 — GBM은 깊을수록 내려가고 RF는 깊을수록 오름

- `1 0.9650       1.0 0.9653` — 깊이 1 · 둘 다 0.965
- `3 0.9232       1.0 0.9722` — 깊이 3 · GBM은 내려가고 RF는 오름
- `None 0.9020       1.0 0.9791` — 끝까지 · GBM 0.902(가장 낮음) · RF 0.979(가장 높음)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Boosting · 얕은 트리 (Q2 · Q4-1)</td>
<td style="text-align: left;">각 트리는 편향을 조금씩 줄이는 약한 보정 · 깊으면 첫 스텝에서 과적합 → 잔차가 노이즈</td>
<td style="text-align: left;">깊이 1 ~ 3 · 작은 학습률</td>
</tr>
<tr>
<td style="text-align: left;">RF · 깊은 트리 (Q2 · Q4-2)</td>
<td style="text-align: left;">각 트리는 분산이 큰 강한 모델 · 독립 평균이 분산을 줄임</td>
<td style="text-align: left;">깊게 · 너무 얕으면 평균해도 편향이 남음</td>
</tr>
<tr>
<td style="text-align: left;">Q3 · 기억 vs 서로 모름</td>
<td style="text-align: left;">Boosting은 앞 모델의 잔차 · 가중치를 기억 · Bagging은 독립</td>
<td style="text-align: left;">기억은 편향 ↓ 이지만 노이즈도 기억</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - GB train이 깊이와 상관없이 1.0 → 과적합 여부는 train이 아니라 CV로 판단
> - Wine 142병 · 한 데이터의 결과 · 원본 숙제 Q5(Hessian)는 노이즈가 많거나 데이터가 적으면 2차 미분까지 맞추는 정밀함이 과적합으로 이어질 수 있다는 관점에서 생각

> ✅ **한 줄 정리** · Boosting은 순차로 잔차를 보정하므로 얕은 트리가 맞고, Random Forest는 독립 트리를 평균하므로 깊은 트리를 써도 분산이 줄어든다.

**딥러닝 코드에서는**

- 작은 보정을 많이(부스팅) vs 강한 모델을 여러 개 평균(앙상블) — 신경망에서도 같은 설계 선택

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">학습 목표</th>
<th style="text-align: left;">할 수 있게 된 것</th>
<th style="text-align: left;">확인 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1. 회귀 트리 · 잔차</td>
<td style="text-align: left;">R² 0.864 · 잎 8 · 잔차 보정 MSE 0.481 → 0.244</td>
<td style="text-align: left;">1-1 · 1-2</td>
</tr>
<tr>
<td style="text-align: left;">2. AdaBoost</td>
<td style="text-align: left;">stump 0.627 → 30개 0.951 · lr 0.3 × 200 · test 0.972</td>
<td style="text-align: left;">2-1 ~ 2-3</td>
</tr>
<tr>
<td style="text-align: left;">3. Gradient Boosting</td>
<td style="text-align: left;">lr 1.1 발산 · train 안 CV 0.9719 · test 1.000 (원본 절차의 누수 수정)</td>
<td style="text-align: left;">3-1 ~ 3-3</td>
</tr>
<tr>
<td style="text-align: left;">4. 손실 함수</td>
<td style="text-align: left;">squared 데이터 RMSE 0.039인데 진짜 sin과 0.542 · absolute 0.119</td>
<td style="text-align: left;">4-1 · 4-2</td>
</tr>
<tr>
<td style="text-align: left;">5. 라이브러리 · 깊이</td>
<td style="text-align: left;">bin 10이 0.9717 · 원-핫 0.95 vs 네이티브 0.90 · GBM 깊을수록 ↓ · RF 깊을수록 ↑</td>
<td style="text-align: left;">5장 · 6</td>
</tr>
</tbody>
</table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">등급</th>
<th style="text-align: left;">주제</th>
<th style="text-align: left;">반드시 기억할 것</th>
<th style="text-align: left;">딥러닝 코드에서</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">Boosting</td>
<td style="text-align: left;">순차로 잔차 · 가중치 보정 → 편향 ↓</td>
<td style="text-align: left;">잔차 연결 · 반복 업데이트</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">회귀 트리</td>
<td style="text-align: left;">잎 = 평균 · 계단 함수</td>
<td style="text-align: left;">회귀 출력층</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">AdaBoost</td>
<td style="text-align: left;">틀린 샘플 가중치 ↑ · stump</td>
<td style="text-align: left;">hard example mining</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">Gradient Boosting</td>
<td style="text-align: left;">손실의 기울기(잔차)를 따라 트리 추가</td>
<td style="text-align: left;">경사하강</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">learning_rate × n_estimators</td>
<td style="text-align: left;">짝으로 움직임 · 함께 탐색</td>
<td style="text-align: left;">학습률 × epoch</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">평가 계약</td>
<td style="text-align: left;">CV는 train 안 · test는 선택에 쓰지 않음</td>
<td style="text-align: left;">val 튜닝 · test 1회</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">손실 함수</td>
<td style="text-align: left;">이상치면 absolute · huber · 판단은 깨끗한 기준으로</td>
<td style="text-align: left;"><code>HuberLoss</code> · <code>L1Loss</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>B</code></td>
<td style="text-align: left;">XGBoost · LightGBM · CatBoost</td>
<td style="text-align: left;">규제 · 히스토그램 · 범주형 자동</td>
<td style="text-align: left;">표 데이터 기준선</td>
</tr>
<tr>
<td style="text-align: left;"><code>B</code></td>
<td style="text-align: left;">max_bins</td>
<td style="text-align: left;">표현력 vs 안정성</td>
<td style="text-align: left;">양자화</td>
</tr>
<tr>
<td style="text-align: left;"><code>B</code></td>
<td style="text-align: left;">얕은 vs 깊은 트리</td>
<td style="text-align: left;">부스팅은 얕게 · RF는 깊게</td>
<td style="text-align: left;">작은 보정 vs 강한 모델 평균</td>
</tr>
</tbody>
</table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">🎯 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">회귀 트리 잎은 무엇을 내나?</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">잔차를 순차로 보정하면?</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">stump 하나와 30개 AdaBoost의 CV는?</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">학습률이 작으면 무엇이 더 필요한가?</td>
</tr>
<tr>
<td style="text-align: left;">2-3</td>
<td style="text-align: left;">고른 AdaBoost의 test는?</td>
</tr>
<tr>
<td style="text-align: left;">3-1</td>
<td style="text-align: left;">학습률이 너무 크면?</td>
</tr>
<tr>
<td style="text-align: left;">3-2</td>
<td style="text-align: left;">원본 GridSearch의 문제는?</td>
</tr>
<tr>
<td style="text-align: left;">3-3</td>
<td style="text-align: left;">train 안에서 고른 GBM의 test는?</td>
</tr>
<tr>
<td style="text-align: left;">4-1</td>
<td style="text-align: left;">3품종에 exponential 손실은?</td>
</tr>
<tr>
<td style="text-align: left;">4-2</td>
<td style="text-align: left;">이상치가 있을 때 학습 RMSE가 가장 낮은 손실이 좋은가?</td>
</tr>
<tr>
<td style="text-align: left;">5-1</td>
<td style="text-align: left;">작은 데이터에서 bin을 줄이면?</td>
</tr>
<tr>
<td style="text-align: left;">5-2</td>
<td style="text-align: left;">범주형 열을 다루는 두 방식은?</td>
</tr>
<tr>
<td style="text-align: left;">5-3</td>
<td style="text-align: left;">부스팅 3종 CV는?</td>
</tr>
</tbody>
</table>

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `AdaBoostClassifier(estimator=stump, n_estimators=100, learning_rate=0.1, random_state=42)`의 train 안 CV를 소수 셋째 자리로 출력
2. 같은 모델을 train에 fit해 학습된 약한 모델 개수(`len(estimators_)`)를 출력

In [16]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. 1-2에서 보정 학습률을 0.5 → 1.0으로 바꾸면 5스텝 뒤 MSE는?
2. 4-2에서 huber의 `alpha`를 0.5로 낮추면 진짜 sin과의 RMSE는?
3. 6장에서 GBM 학습률을 0.01로 낮추면 깊이 3의 CV는?

In [17]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">그 잎에 속한 데이터의 평균 · 깊이 3이면 최대 8칸 계단</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">MSE 0.481 → 0.244 (5스텝 · 학습률 0.5)</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">0.6268 → 0.9512</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">스텝 수 (0.01 × 300도 0.838)</td>
</tr>
<tr>
<td style="text-align: left;">2-3</td>
<td style="text-align: left;">0.972</td>
</tr>
<tr>
<td style="text-align: left;">3-1</td>
<td style="text-align: left;">최소점을 지나쳐 발산 (1.1 → loss 958)</td>
</tr>
<tr>
<td style="text-align: left;">3-2</td>
<td style="text-align: left;">X, y 전체로 탐색해 test가 선택에 섞임</td>
</tr>
<tr>
<td style="text-align: left;">3-3</td>
<td style="text-align: left;">1.000 (36병 · 1병 = 0.028)</td>
</tr>
<tr>
<td style="text-align: left;">4-1</td>
<td style="text-align: left;">ValueError · 이진 분류 전용</td>
</tr>
<tr>
<td style="text-align: left;">4-2</td>
<td style="text-align: left;">아님 · squared는 이상치를 쫓음 · 진짜 sin과는 absolute가 가장 가까움</td>
</tr>
<tr>
<td style="text-align: left;">5-1</td>
<td style="text-align: left;">손해가 없거나 규제 효과 (bin 10이 0.9717로 최고)</td>
</tr>
<tr>
<td style="text-align: left;">5-2</td>
<td style="text-align: left;">원-핫 + Pipeline · 네이티브 범주 처리(CatBoost · HistGB)</td>
</tr>
<tr>
<td style="text-align: left;">5-3</td>
<td style="text-align: left;">GBM 0.923 · XGBoost 0.958 · HistGB 0.958</td>
</tr>
</tbody>
</table>

#### 코드 문제 답

```python
ada_q = AdaBoostClassifier(estimator=stump, n_estimators=100, learning_rate=0.1, random_state=42)
print(round(cross_val_score(ada_q, X_train, y_train, cv=cv).mean(), 3))  # 0.916
print(len(ada_q.fit(X_train, y_train).estimators_))  # 100
```

#### 변형 문제 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">문제</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">0.1222 — 보폭이 커서 5스텝에 더 빨리 줄지만 노이즈도 더 빨리 맞춤</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">0.175 — alpha를 낮추면 더 많은 오차를 절댓값처럼 다뤄 이상치에 덜 끌림</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">0.9022 — 0.9232보다 낮음 · 학습률을 낮추면 스텝 200으로는 부족 (짝으로 탐색)</td>
</tr>
</tbody>
</table>

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">틀린 내용</th>
<th style="text-align: left;">틀린 이유</th>
<th style="text-align: left;">올바른 원칙</th>
<th style="text-align: left;">재발 방지 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">(예) 이상치 데이터에서 RMSE가 가장 낮은 squared_error를 고름</td>
<td style="text-align: left;">학습 데이터 RMSE를 성능으로 착각</td>
<td style="text-align: left;">이상치가 섞인 데이터가 아니라 깨끗한 기준(진짜 함수 · 검증)으로 판단</td>
<td style="text-align: left;">이 RMSE는 무엇과의 거리인가?</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
</tbody>
</table>

---
## 면접 30초 답변

**Q1. Bagging과 Boosting의 차이는 무엇인가요?**
> - Bagging은 같은 모델을 bootstrap 표본마다 독립적으로 학습해 평균내 분산을 줄이고, Boosting은 앞 모델이 틀린 부분을 다음 모델이 순차로 보정해 편향을 줄입니다.
> - 그래서 RF는 깊은 트리를, 부스팅은 얕은 트리와 작은 학습률을 씁니다.
> - Wine에서도 GBM은 깊이 1이 0.965로 가장 좋았고 RF는 깊이 제한 없음이 0.979로 가장 좋았습니다.

**Q2. AdaBoost와 Gradient Boosting은 어떻게 다른가요?**
> - AdaBoost는 틀린 샘플의 가중치를 올려 다음 약한 학습기가 그 샘플에 집중하게 하고, 모델마다 성능에 따른 가중치로 투표합니다.
> - Gradient Boosting은 손실 함수의 기울기, 즉 잔차를 다음 트리가 맞추게 해서 손실을 직접 줄입니다.
> - 손실을 바꿀 수 있어 회귀와 다중 분류, 이상치에 강한 손실까지 다룹니다.

**Q3. 부스팅의 하이퍼파라미터는 어떻게 고르나요?**
> - 학습률과 스텝 수는 짝으로 움직여서 함께 탐색합니다.
> - 학습률이 작으면 안정적이지만 스텝이 더 필요하고, 크면 빠르지만 과적합되거나 발산할 수 있습니다.
> - 탐색은 test를 봉인한 채 train 안의 교차검증으로 하고, 고른 조합으로 test를 한 번만 평가합니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">함수·문법</th>
<th style="text-align: left;">소속</th>
<th style="text-align: left;">역할</th>
<th style="text-align: left;">코드에서 사용한 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>DecisionTreeRegressor</code> · <code>export_text</code></td>
<td style="text-align: left;">sklearn.tree</td>
<td style="text-align: left;">회귀 트리 · 글자로 보기</td>
<td style="text-align: left;">1-1 <code>DecisionTreeRegressor(max_depth=3, …)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>AdaBoostClassifier(estimator=, n_estimators=, learning_rate=)</code></td>
<td style="text-align: left;">sklearn.ensemble</td>
<td style="text-align: left;">틀린 샘플 가중치 부스팅</td>
<td style="text-align: left;">2-1 <code>AdaBoostClassifier(estimator=stump, …)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>GridSearchCV</code> · <code>pivot_table</code></td>
<td style="text-align: left;">sklearn · pandas</td>
<td style="text-align: left;">조합 탐색 · 히트맵 표</td>
<td style="text-align: left;">2-2 <code>res.pivot_table(index=…, columns=…)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>GradientBoostingClassifier(subsample=, max_depth=)</code></td>
<td style="text-align: left;">sklearn.ensemble</td>
<td style="text-align: left;">기울기 부스팅</td>
<td style="text-align: left;">3-2 <code>GridSearchCV(GradientBoostingClassifier(…), …)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>GradientBoostingRegressor(loss=)</code></td>
<td style="text-align: left;">sklearn.ensemble</td>
<td style="text-align: left;">회귀 손실 선택</td>
<td style="text-align: left;">4-2 <code>loss=loss</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>HistGradientBoostingClassifier(max_bins=, categorical_features=)</code></td>
<td style="text-align: left;">sklearn.ensemble</td>
<td style="text-align: left;">히스토그램 · 네이티브 범주</td>
<td style="text-align: left;">5-1 <code>max_bins=b</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>XGBClassifier(eval_metric=)</code></td>
<td style="text-align: left;">xgboost</td>
<td style="text-align: left;">XGBoost</td>
<td style="text-align: left;">5-3 <code>XGBClassifier(n_estimators=200, …)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>ColumnTransformer</code> · <code>OneHotEncoder</code> · <code>Pipeline</code></td>
<td style="text-align: left;">sklearn</td>
<td style="text-align: left;">범주형 원-핫</td>
<td style="text-align: left;">5-2 <code>ColumnTransformer([("num", …), ("cat", …)])</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>RandomForestClassifier(max_depth=)</code></td>
<td style="text-align: left;">sklearn.ensemble</td>
<td style="text-align: left;">깊이 비교</td>
<td style="text-align: left;">6 <code>RandomForestClassifier(n_estimators=300, max_depth=d, …)</code></td>
</tr>
</tbody>
</table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">✅ 할 것</th>
<th style="text-align: left;">⚠️ 하지 말 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">CV는 test를 뺀 train 안에서</td>
<td style="text-align: left;">전체 데이터로 CV한 뒤 같은 test 보고</td>
</tr>
<tr>
<td style="text-align: left;">학습률 × 스텝을 함께 탐색</td>
<td style="text-align: left;">한쪽만 바꿔 결론</td>
</tr>
<tr>
<td style="text-align: left;">이상치면 깨끗한 기준으로 손실 판단</td>
<td style="text-align: left;">학습 RMSE 최저 = 최고 모델</td>
</tr>
<tr>
<td style="text-align: left;">부스팅은 얕은 트리 · 작은 학습률</td>
<td style="text-align: left;">부스팅에 깊은 트리</td>
</tr>
<tr>
<td style="text-align: left;">라이브러리는 같은 계약으로 비교</td>
<td style="text-align: left;">유명한 라이브러리가 항상 이긴다고 가정</td>
</tr>
</tbody>
</table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">짝</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">결론</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Bagging vs Boosting</td>
<td style="text-align: left;">병렬 · 분산 ↓ vs 순차 · 편향 ↓</td>
<td style="text-align: left;">깊은 트리 vs 얕은 트리</td>
</tr>
<tr>
<td style="text-align: left;">AdaBoost vs GBM</td>
<td style="text-align: left;">샘플 가중치 vs 손실의 기울기</td>
<td style="text-align: left;">GBM이 더 일반적</td>
</tr>
<tr>
<td style="text-align: left;">squared vs absolute</td>
<td style="text-align: left;">평균 · 이상치 민감 vs 중앙값 · 이상치 강함</td>
<td style="text-align: left;">이상치 유무로</td>
</tr>
<tr>
<td style="text-align: left;">원-핫 vs 네이티브 범주</td>
<td style="text-align: left;">차원 증가 · 모든 모델 vs 인코딩 없음 · 일부 라이브러리</td>
<td style="text-align: left;">같은 데이터로 비교</td>
</tr>
<tr>
<td style="text-align: left;">learning_rate 큼 vs 작음</td>
<td style="text-align: left;">빠름 · 과적합 · 발산 vs 안정 · 스텝 더 필요</td>
<td style="text-align: left;">짝으로 탐색</td>
</tr>
</tbody>
</table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">언제</th>
<th style="text-align: left;">할 일</th>
<th style="text-align: left;">걸리는 시간</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">내일</td>
<td style="text-align: left;">기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td>
<td style="text-align: left;">15분</td>
</tr>
<tr>
<td style="text-align: left;">3일 뒤</td>
<td style="text-align: left;">변형 문제 3개 · 1-2 잔차 보정을 손으로 한 스텝</td>
<td style="text-align: left;">20분</td>
</tr>
<tr>
<td style="text-align: left;">7일 뒤</td>
<td style="text-align: left;">원본 숙제 Q1 ~ Q5를 6장 표 · 면접 답변을 근거로 글로 쓰기</td>
<td style="text-align: left;">30분</td>
</tr>
<tr>
<td style="text-align: left;">14일 뒤</td>
<td style="text-align: left;">오답 노트에 남은 문제만 · 머신러닝 파일별 10권의 ✅ 줄만 이어서 읽기</td>
<td style="text-align: left;">20분</td>
</tr>
</tbody>
</table>

---
## 부록

### 이 노트북을 만든 방법

- 원본 노트북의 셀 99개를 1 ~ 5장 · 13개 섹션으로 나누고, 마지막 숙제(Q1 ~ Q5)는 6장 실험 · 면접 답변으로
- 원본의 CV · GridSearch가 test를 포함한 전체 데이터로 돌던 부분은 test를 먼저 봉인하고 train 안 CV로 바꿈 (숫자가 원본과 다름)

- - lightgbm · catboost
- graphviz가 이 환경에 없어 sklearn HistGradientBoosting
- export_text로 같은 개념을 실행하고 원본 결과는 caution에 보존
- 원본 슬라이드 png 18장(폴더에 없음) · plotly 그림 9개는 장 그림 · 비교 표로 · 원본 마지막 `print('hoho')` 테스트 셀은 생략

### 읽는 법

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">표시</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code> · <code>B</code></td>
<td style="text-align: left;"><strong>A</strong> 딥러닝 코드에 바로 나오는 필수 · <strong>B</strong> 알아두면 되는 것</td>
</tr>
<tr>
<td style="text-align: left;">🎯</td>
<td style="text-align: left;">이번 섹션에서 답해야 할 질문</td>
</tr>
<tr>
<td style="text-align: left;">💡</td>
<td style="text-align: left;">쉬운 설명 · 비유</td>
</tr>
<tr>
<td style="text-align: left;">⚠️</td>
<td style="text-align: left;">자주 하는 실수</td>
</tr>
<tr>
<td style="text-align: left;">✅</td>
<td style="text-align: left;">기억할 결론 = 🎯 질문의 답</td>
</tr>
</tbody>
</table>

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">원본</th>
<th style="text-align: left;">내용</th>
<th style="text-align: left;">이 노트북</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">셀 1 ~ 5</td>
<td style="text-align: left;">목표 · Bagging vs Boosting 복습</td>
<td style="text-align: left;">0. 원본 목표</td>
</tr>
<tr>
<td style="text-align: left;">셀 2</td>
<td style="text-align: left;">준비</td>
<td style="text-align: left;">0. 준비 코드</td>
</tr>
<tr>
<td style="text-align: left;">셀 6 ~ 12</td>
<td style="text-align: left;">회귀 트리 · graphviz · 계단 그림</td>
<td style="text-align: left;">1-1 · 1장 그림</td>
</tr>
<tr>
<td style="text-align: left;">셀 13 ~ 16</td>
<td style="text-align: left;">Boosting 아이디어 · 핵심 로직</td>
<td style="text-align: left;">1-2</td>
</tr>
<tr>
<td style="text-align: left;">셀 17 ~ 26</td>
<td style="text-align: left;">AdaBoost · stump · n_estimators 곡선</td>
<td style="text-align: left;">2-1</td>
</tr>
<tr>
<td style="text-align: left;">셀 27 ~ 29</td>
<td style="text-align: left;">learning_rate × n_estimators 히트맵</td>
<td style="text-align: left;">2-2 · 2장 그림</td>
</tr>
<tr>
<td style="text-align: left;">셀 30 ~ 33</td>
<td style="text-align: left;">최종 AdaBoost · 가중치의 함정</td>
<td style="text-align: left;">2-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 34 ~ 46</td>
<td style="text-align: left;">GB 개념 · 차이 · 경사하강 시각화</td>
<td style="text-align: left;">3-1 · 3장 그림</td>
</tr>
<tr>
<td style="text-align: left;">셀 47 ~ 49</td>
<td style="text-align: left;">GB GridSearch · 히트맵</td>
<td style="text-align: left;">3-2</td>
</tr>
<tr>
<td style="text-align: left;">셀 50</td>
<td style="text-align: left;">GB 최종 평가</td>
<td style="text-align: left;">3-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 51 ~ 54</td>
<td style="text-align: left;">분류 손실</td>
<td style="text-align: left;">4-1</td>
</tr>
<tr>
<td style="text-align: left;">셀 55 ~ 59</td>
<td style="text-align: left;">회귀 손실 · 이상치 · 선택 가이드</td>
<td style="text-align: left;">4-2 · 4장 그림</td>
</tr>
<tr>
<td style="text-align: left;">셀 60 ~ 69</td>
<td style="text-align: left;">XGBoost · LightGBM · bin</td>
<td style="text-align: left;">5-1 · 5장 그림</td>
</tr>
<tr>
<td style="text-align: left;">셀 77 ~ 85</td>
<td style="text-align: left;">CatBoost · 범주형</td>
<td style="text-align: left;">5-2</td>
</tr>
<tr>
<td style="text-align: left;">셀 70 ~ 76 · 86 ~ 88</td>
<td style="text-align: left;">라이브러리 비교 · 사용 예시 · 4종 비교 · 선택 가이드</td>
<td style="text-align: left;">5-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 89 ~ 91</td>
<td style="text-align: left;">Bagging vs Boosting 정리</td>
<td style="text-align: left;">0. 원본 표 · 헷갈리는 짝</td>
</tr>
<tr>
<td style="text-align: left;">셀 92 ~ 97</td>
<td style="text-align: left;">숙제 Q1 ~ Q5</td>
<td style="text-align: left;">6. 종합 실습 · 면접</td>
</tr>
<tr>
<td style="text-align: left;">셀 0 · 98</td>
<td style="text-align: left;">빈 셀 · print('hoho')</td>
<td style="text-align: left;">생략</td>
</tr>
</tbody>
</table>

### 전체 코드

1 ~ 6장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

In [18]:
# ---------- 1-1 ----------
np.random.seed(42)                                                                # 원본 셀 8
X_reg = np.sort(np.random.rand(80, 1) * 10, axis=0)                               # x 0 ~ 10
y_reg = np.sin(X_reg).ravel() + np.random.normal(0, 0.2, 80)                      # sin + 노이즈
reg_tree = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_reg, y_reg)  # 깊이 3
p = reg_tree.predict(X_reg)                                                       # 학습 데이터 예측
print(f"R² Score : {r2_score(y_reg, p):.3f}")                                     # 원본 형식
print(f"RMSE     : {np.sqrt(mean_squared_error(y_reg, p)):.3f}")                  # 원본 형식
print("잎 개수:", reg_tree.get_n_leaves(), "· 서로 다른 예측값:", len(np.unique(p)))  # 계단 수
# 첫 질문 (원본은 graphviz)
print(export_text(reg_tree, feature_names=["x"], decimals=2).splitlines()[0])


# ---------- 1-2 ----------
pred = np.full_like(y_reg, y_reg.mean())                                # 1. 첫 예측 = 평균
print(f"출발(평균만) MSE {np.mean((y_reg - pred) ** 2):.4f}")           # 기준
for step in range(1, 6):                                                # 순차 5번
    resid = y_reg - pred                                                # 2. 잔차 = 실제 - 예측
    stump = DecisionTreeRegressor(max_depth=1, random_state=42).fit(X_reg, resid)  # 3. 잔차를 학습
    pred = pred + 0.5 * stump.predict(X_reg)                            # 4. 학습률 0.5만큼 보정
    print(f"step {step}: 잔차 MSE {np.mean((y_reg - pred) ** 2):.4f}")  # 줄어드는지

R² Score : 0.864
RMSE     : 0.256
잎 개수: 8 · 서로 다른 예측값: 8
|--- x <= 2.65
출발(평균만) MSE 0.4810
step 1: 잔차 MSE 0.3934
step 2: 잔차 MSE 0.3457
step 3: 잔차 MSE 0.2997
step 4: 잔차 MSE 0.2718
step 5: 잔차 MSE 0.2441


In [19]:
# ---------- 2-1 ----------
stump = DecisionTreeClassifier(max_depth=1, random_state=42)                   # 약한 학습기
print("stump 하나 CV:", round(cross_val_score(stump, X_train, y_train, cv=cv).mean(), 4))  # 출발점
rows = []                                                                      # n마다
for n in [10, 30, 50, 100, 200, 300]:                                          # 원본 셀 25
    ada = AdaBoostClassifier(estimator=stump, n_estimators=n,                  # stump n개
                             learning_rate=0.5, random_state=42)               # 보정 강도 0.5
    s = cross_val_score(ada, X_train, y_train, cv=cv, n_jobs=-1).mean()        # train 안에서만
    tr = ada.fit(X_train, y_train).score(X_train, y_train)                     # 외운 문제
    rows.append([n, tr, s])                                                    # 한 줄
table = pd.DataFrame(rows, columns=["n_estimators", "train", "CV(train 안)"])  # 비교표
print(table.round(4).to_string(index=False))                                   # 소수 넷째 자리


# ---------- 2-2 ----------
lr_list, n_list = [0.3, 0.2, 0.1, 0.05, 0.01], [30, 50, 100, 200, 300]          # 원본 셀 28
base = AdaBoostClassifier(estimator=stump, random_state=42)                     # 같은 stump
ada_grid = GridSearchCV(base, {"learning_rate": lr_list, "n_estimators": n_list},  # 25조합
                        cv=cv, n_jobs=-1).fit(X_train, y_train)
res = pd.DataFrame(ada_grid.cv_results_)                                        # 조합별 결과
heat = res.pivot_table(index="param_learning_rate",                             # 행 = 학습률
                       columns="param_n_estimators", values="mean_test_score")  # 열 = 스텝 수
print(heat.sort_index(ascending=False).round(3).to_string())                    # 원본 히트맵의 숫자
print("Best by CV:", ada_grid.best_params_, round(ada_grid.best_score_, 3))     # 최고
print("공동 1위:", int((res["rank_test_score"] == 1).sum()), "개")              # 동점 확인


# ---------- 2-3 ----------
y_pred = ada_grid.best_estimator_.predict(X_test)                              # test 첫 사용
print(f"[AdaBoost Best] Test Accuracy: {accuracy_score(y_test, y_pred):.3f}")  # 원본 형식
print(classification_report(y_test, y_pred, target_names=wine.target_names, digits=3))  # 클래스별

stump 하나 CV: 0.6268


 n_estimators  train  CV(train 안)
           10 0.9789       0.8808
           30 1.0000       0.9512
           50 1.0000       0.9441
          100 1.0000       0.9510
          200 1.0000       0.9441
          300 1.0000       0.9372


param_n_estimators     30     50     100    200    300
param_learning_rate                                   
0.30                 0.930  0.923  0.937  0.951  0.951
0.20                 0.888  0.916  0.937  0.937  0.944
0.10                 0.846  0.867  0.916  0.930  0.937
0.05                 0.831  0.838  0.860  0.923  0.930
0.01                 0.627  0.704  0.817  0.846  0.838
Best by CV: {'learning_rate': 0.3, 'n_estimators': 200} 0.951
공동 1위: 2 개
[AdaBoost Best] Test Accuracy: 0.972
              precision    recall  f1-score   support

     class_0      0.923     1.000     0.960        12
     class_1      1.000     0.929     0.963        14
     class_2      1.000     1.000     1.000        10

    accuracy                          0.972        36
   macro avg      0.974     0.976     0.974        36
weighted avg      0.974     0.972     0.972        36



In [20]:
# ---------- 3-1 ----------
for lr, steps in [(0.01, 80), (0.1, 20), (0.3, 10), (1.1, 10)]:      # 원본 3개 + 발산 예시
    x = 5.0                                                          # 출발점
    for _ in range(steps):                                           # 걸음마다
        x -= lr * 2 * x                                              # 기울기(2x)의 반대 방향으로
    print(f"lr={lr}: {steps}걸음 뒤 x={x:.4f} · loss={x ** 2:.4f}")  # 도착


# ---------- 3-2 ----------
param_grid = {"learning_rate": [0.3, 0.1, 0.05, 0.01],                      # 원본 셀 48
              "n_estimators": [50, 100, 200, 300],                          # 스텝 수
              "subsample": [1.0, 0.8], "max_depth": [3, 2]}                 # 4 · 4 · 2 · 2 = 64조합
gs = GridSearchCV(GradientBoostingClassifier(random_state=42), param_grid,  # 원본은 X, y 전체로 fit
                  cv=cv, n_jobs=-1).fit(X_train, y_train)
print("best_params_ :", gs.best_params_)                                    # 고른 조합
print("cv best_score:", round(gs.best_score_, 4))                           # 고를 때 쓴 점수
print("공동 1위:", int((gs.cv_results_["rank_test_score"] == 1).sum()), "/ 64")  # 동점


# ---------- 3-3 ----------
y_pred_gb = gs.best_estimator_.predict(X_test)                          # test 첫 사용
print(f"Final Test Accuracy: {accuracy_score(y_test, y_pred_gb):.3f}")  # 원본 형식
print("틀린 병:", int((y_pred_gb != y_test).sum()), "/ 36")             # 1병 = 0.028

lr=0.01: 80걸음 뒤 x=0.9932 · loss=0.9865
lr=0.1: 20걸음 뒤 x=0.0576 · loss=0.0033
lr=0.3: 10걸음 뒤 x=0.0005 · loss=0.0000
lr=1.1: 10걸음 뒤 x=30.9587 · loss=958.4400


best_params_ : {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 100, 'subsample': 0.8}
cv best_score: 0.9719
공동 1위: 1 / 64
Final Test Accuracy: 1.000
틀린 병: 0 / 36


In [21]:
# ---------- 4-1 ----------
gb_loss = GradientBoostingClassifier(                                     # 원본 셀 54
    loss="log_loss", n_estimators=200, learning_rate=0.1,                 # 분류 기본 손실
    max_depth=2, subsample=0.8, random_state=42)                          # 얕은 트리 · 행 80%
s = cross_val_score(gb_loss, X_train, y_train, cv=cv)                     # train 안
print(f"[분류] loss='log_loss' · CV Accuracy: {s.mean():.3f} ± {s.std():.3f}")  # 원본 형식
try:                                                                      # 원본은 주석으로만 경고
    GradientBoostingClassifier(loss="exponential").fit(X_train, y_train)  # 3클래스
except ValueError as e:                                                   # sklearn이 막음
    print("exponential:", str(e)[:70])                                    # 이진 전용


# ---------- 4-2 ----------
y_out = y_reg.copy()                                                   # 원본 셀 56
y_out[10], y_out[50] = 5.0, -4.0                                       # 이상치 2개
X_plot = np.linspace(0, 10, 500).reshape(-1, 1)                        # 진짜 sin과 비교할 x
for loss in ["squared_error", "absolute_error", "huber"]:              # 원본 셀 58
    gb = GradientBoostingRegressor(loss=loss, n_estimators=200, learning_rate=0.1,  # 같은 설정
                                   max_depth=3, random_state=42).fit(X_reg, y_out)
    rmse_data = np.sqrt(mean_squared_error(y_out, gb.predict(X_reg)))  # 이상치 포함 데이터와의 거리
    # 진짜 함수와의 거리
    rmse_true = np.sqrt(mean_squared_error(np.sin(X_plot).ravel(), gb.predict(X_plot)))
    print(f"{loss:<15s} RMSE(이상치 포함 데이터) {rmse_data:.3f}",     # 학습 데이터 기준
          f"· RMSE(진짜 sin) {rmse_true:.3f}")                         # 진짜 함수 기준

[분류] loss='log_loss' · CV Accuracy: 0.958 ± 0.034
exponential: loss='exponential' is only suitable for a binary classification proble
squared_error   RMSE(이상치 포함 데이터) 0.039 · RMSE(진짜 sin) 0.542


absolute_error  RMSE(이상치 포함 데이터) 0.637 · RMSE(진짜 sin) 0.119


huber           RMSE(이상치 포함 데이터) 0.136 · RMSE(진짜 sin) 0.530


In [22]:
from sklearn.compose import ColumnTransformer                    # 열마다 다른 전처리
from sklearn.pipeline import Pipeline                            # 전처리 + 모델
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # 원-핫 · 스케일링


# ---------- 5-1 ----------
rows = []                                                                     # bin마다
for b in [10, 31, 63, 127, 255]:                                              # 원본 셀 66의 후보
    # 히스토그램 부스팅
    hgb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, max_bins=b,
                                         max_leaf_nodes=31, random_state=42)  # 잎 31 (leaf-wise)
    rows.append([b, cross_val_score(hgb, X_train, y_train, cv=cv).mean()])    # train 안 CV
# 표
print(pd.DataFrame(rows, columns=["max_bins", "CV(train 안)"]).round(4).to_string(index=False))


# ---------- 5-2 ----------

np.random.seed(42)                                                           # 원본 셀 80
n = 200                                                                      # 행 수
df = pd.DataFrame({"color": np.random.choice(["red", "green", "blue"], n),   # 범주형
                   "size": np.random.choice(["small", "medium", "large"], n),  # 범주형
                   "weight": np.random.normal(50, 10, n).round(1),           # 수치형
                   "price": np.random.normal(1000, 300, n).round(0)})        # 수치형
df["target"] = ((df["color"] == "red") & (df["weight"] > 50)).astype(int)    # 빨강 + 무거움 → 1
noise_idx = np.random.choice(n, size=15, replace=False)                      # 노이즈 15개
df.loc[noise_idx, "target"] = 1 - df.loc[noise_idx, "target"]                # 뒤집기
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(df.drop(columns="target"), df["target"],  # 8 : 2
                                              test_size=0.2, stratify=df["target"], random_state=42)
pre = ColumnTransformer([("num", StandardScaler(), ["weight", "price"]),     # 원본 셀 82
                         ("cat", OneHotEncoder(handle_unknown="ignore"), ["color", "size"])])
xgb_c = XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=3,      # 원본 셀 83
                      eval_metric="logloss", random_state=42)                # 이진 손실
pipe = Pipeline([("pre", pre), ("model", xgb_c)]).fit(Xc_tr, yc_tr)          # 원-핫 → XGBoost
acc_oh = accuracy_score(yc_te, pipe.predict(Xc_te))                          # 원-핫 방식 정확도
cat_t = {"color": "category", "size": "category"}  # 범주 타입 지정 (cat_features에 해당)
# 인코딩 없이
hgb_c = HistGradientBoostingClassifier(categorical_features="from_dtype", max_iter=200,
                                       learning_rate=0.1, max_depth=3, random_state=42)
hgb_c.fit(Xc_tr.astype(cat_t), yc_tr)                                        # 범주 열을 그대로
acc_nat = accuracy_score(yc_te, hgb_c.predict(Xc_te.astype(cat_t)))          # 네이티브 방식 정확도
print("데이터:", df.shape, "· 타깃", df["target"].value_counts().to_dict())  # 200 · 157 : 43
print(f"원-핫 + XGBoost Accuracy: {acc_oh:.3f}",                             # 방식 1
      f"· 네이티브 범주(HistGB) Accuracy: {acc_nat:.3f}")                    # 방식 2


# ---------- 5-3 ----------
gbm = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,  # sklearn 기본 부스팅
                                 max_depth=3, random_state=42)  # 깊이 3
models = {"GBM": gbm,                                           # 세 모델 같은 200스텝 · 학습률 0.1
          "XGBoost": XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=3,  # 원본 셀 87
                                   eval_metric="mlogloss", random_state=42),
          "HistGB": HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1,  # LightGBM 방식
                                                   max_leaf_nodes=31, random_state=42)}
for name, m in models.items():                                  # 같은 계약
    s = cross_val_score(m, X_train, y_train, cv=cv, n_jobs=-1)  # train 안
    print(f"{name:<8s} CV {s.mean():.3f} ± {s.std():.3f}")      # 평균 · 흔들림

 max_bins  CV(train 안)
       10       0.9717
       31       0.9512
       63       0.9650
      127       0.9581
      255       0.9581
데이터: (200, 5) · 타깃 {0: 157, 1: 43}
원-핫 + XGBoost Accuracy: 0.950 · 네이티브 범주(HistGB) Accuracy: 0.900


GBM      CV 0.923 ± 0.070


XGBoost  CV 0.958 ± 0.051
HistGB   CV 0.958 ± 0.034


In [23]:
# ---------- 6. 종합 실습 ----------
# 깊이만 바꿔 GBM vs RF (train 안 CV · test는 안 씀)
rows = []                                                                         # 깊이마다
for d in [1, 3, None]:                                                            # 1 · 3 · 끝까지
    gbd = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,         # 순차 · 잔차 보정
                                     # GBM은 None 대신 10
                                     max_depth=d if d else 10, random_state=42)
    rfd = RandomForestClassifier(n_estimators=300, max_depth=d, random_state=42)  # 병렬 · 평균
    gb_cv = cross_val_score(gbd, X_train, y_train, cv=cv, n_jobs=-1).mean()       # GBM CV
    gb_tr = gbd.fit(X_train, y_train).score(X_train, y_train)                     # GBM 외운 문제
    rf_cv = cross_val_score(rfd, X_train, y_train, cv=cv, n_jobs=-1).mean()       # RF CV
    rows.append([str(d), gb_cv, gb_tr, rf_cv])                                    # 한 줄
table = pd.DataFrame(rows, columns=["depth", "GB CV", "GB train", "RF CV"])       # 비교표
print(table.round(4).to_string(index=False))                                      # 소수 넷째 자리

depth  GB CV  GB train  RF CV
    1 0.9650       1.0 0.9653
    3 0.9232       1.0 0.9722
 None 0.9020       1.0 0.9791
